# Does dealer positioning inferred from public SDR prints predict SR3 / ZQ?

**This notebook renders; it does not compute.** The gates are run by
`scripts/run_dealer_ladder_gates.py`, which writes every artifact to
`BT/results/dealer_ladder/*.csv`. That split is the repo convention and it is what makes an
*executed* notebook a reviewable artifact instead of a multi-hour job nobody re-runs.

- Pre-registration and verdict: `docs/superpowers/plans/2026-07-30-dealer-ladder-signal-findings.md`
- Running journal (decisions, timings, constraints): `docs/superpowers/plans/2026-07-29-dealer-ladder-research-journal.md`

**Read the gates in order.** G2 (does the ladder lead measurable futures flow?) runs before any
price test, because a price result whose mechanism was never established is a different and much
weaker claim. A failed gate is a result, reported as such.

**What this is not.** The direction label is a model inference from price-versus-curve-mid, not an
observed counterparty field, and it is uncertified against truth labels — no desk tickets were
available. Every signed number therefore carries the `(2a-1)` attenuation grid, and the object is
described throughout as a *model-labelled D2C flow proxy*, never as dealer inventory.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from BT.dealer_ladder import plots

plots.use_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

RESULTS = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "BT", "results", "dealer_ladder"))


def load(name, **kw):
    """Read a gate artifact, or return an empty frame so a missing stage cannot
    silently look like a passing one."""
    path = os.path.join(RESULTS, f"{name}.csv")
    if not os.path.exists(path):
        print(f"  [missing] {name}.csv — that stage did not produce output")
        return pd.DataFrame()
    return pd.read_csv(path, **kw)


print(f"results dir: {RESULTS}")
print(sorted(os.listdir(RESULTS)) if os.path.isdir(RESULTS) else "  (does not exist yet)")

## Verdict summary

State is shown as **text** beside the swatch, never as colour alone.

In [ ]:
verdicts = load("verdicts")
if len(verdicts):
    display(verdicts[[c for c in ("gate", "pass", "headline") if c in verdicts.columns]])
    plots.verdict_strip(verdicts.to_dict(orient="records"))
else:
    print("no verdicts yet — run scripts/run_dealer_ladder_gates.py")

## G0 — labels and provenance

Which prints are in the signed universe, which gate removed the rest, and does an **independent
mid** agree with the direction call? The independent mid is the Citi Velocity intraday SOFR par
curve, run through the *identical* classifier, so a flip isolates curve disagreement rather than
a difference in rule. Both independent sources are SOFR curves, so FED_FUNDS has no cross-check.

In [ ]:
summary = load("g0_universe_summary", index_col=0)
ladder_tbl = load("g0_exclusion_ladder", index_col=0)
skew = load("g0_skew_by_stratum", index_col=0)
for name, frame in (("universe", summary), ("exclusion ladder", ladder_tbl),
                    ("skew by stratum", skew)):
    if len(frame):
        print(f"-- {name}")
        display(frame)

In [ ]:
flip_conf = load("g0_flip_by_confidence", index_col=0)
flip_type = load("g0_flip_by_trade_type", index_col=0)
skew_ind = load("g0_skew_vs_independent", index_col=0)
if len(flip_conf):
    display(flip_conf)
    plots.flip_rate_bars(flip_conf, group="our_confidence")
if len(flip_type):
    display(flip_type)
if len(skew_ind):
    print("-- PAID share: our mid vs the independent mid, same prints")
    display(skew_ind)

## G1 — arrival integrity

Verified by **differential audit**, not assertion: corrupt every not-yet-visible print and check
the feature does not move. `n_leaking` must be 0 and `max_future_prints` must be > 0 — otherwise
the audit passed vacuously because there was nothing to poison. If this gate fails, everything
downstream is void.

In [ ]:
g1 = load("g1_audits", index_col=0)
if len(g1):
    display(g1)
    if not bool(g1["pass"].all()):
        print("\n  *** LOOK-AHEAD DETECTED — the gates below are not interpretable ***")

## G2 — mechanism ordering (runs before any price test)

Does a signed ladder innovation *precede* measurable signed futures flow? Signed flow is a
**bar-direction** proxy, not Lee-Ready: there is no trade-level data in the repo, only quote
updates. ~39% of SR3 bars close unchanged and contribute zero signed volume, which biases the
measured lead-lag toward zero — the proxy is conservative.

The hypothesised hedge is a **sale** when the ladder is positive (a dealer long
futures-equivalent must sell), so the flow response is expected **opposite** in sign to the
innovation. That is carried explicitly in the `expected_sign` column.

In [ ]:
ll_sum = load("g2_lead_lag_summary_FUTURES", index_col=0)
ll = load("g2_lead_lag_FUTURES", index_col=0)
flow_resp = load("g2_flow_response_FUTURES", index_col=0)
if len(ll_sum):
    print("-- ladder-leads-flow, day-blocked")
    display(ll_sum)
if len(ll):
    print(f"-- per (session, bucket) rows: {len(ll)}")
    display(ll.head(10))
    by_bucket = ll.groupby("bucket")[["lls", "peak_lag_min", "peak_rho"]].mean()
    display(by_bucket)
if len(flow_resp):
    print("-- signed flow response after a large ladder innovation")
    display(flow_resp)

## G3 — circularity battery

The ladder is built on a curve calibrated from the same futures strip the study tries to predict,
so the question is not whether the signal is significant alone but whether it **survives** the
basis, curve shape, momentum, volatility, liquidity, time of day, roll and FOMC proximity. If the
coefficient collapses beside those controls, the effect lives in the basis and the thesis is a
different one.

In [ ]:
race = load("g3_horse_race_FUTURES", index_col=0)
loo = load("g3_leave_one_out_FUTURES", index_col=0)
resid = load("g3_residual_race_FUTURES", index_col=0)
if len(race):
    print("-- horse race (day-clustered errors)")
    display(race)
if len(loo):
    print("-- leave one contract out: a result that needs one contract is one contract's story")
    display(loo)
if len(resid):
    print("-- residualised signal (orthogonalised to the controls)")
    display(resid)

## G4 — the pre-registered price test

**One** locked specification, written down before any statistic existed: FUTURES-space ladder
z-score, 90-minute half-life, expected weighting, whitelisted + curve-clean + on-market strata,
|z| ≥ 1 trigger, 1-hour horizon, SR3 front six, conservative costs, non-overlapping trades.
Pass = mean net > 0 **and** raw t ≥ 3.

In [ ]:
prim = verdicts[verdicts["gate"].astype(str).str.startswith("G4-primary")] if len(verdicts) else pd.DataFrame()
if len(prim):
    display(prim[[c for c in ("gate", "pass", "headline") if c in prim.columns]])
net = load("g5_net_table", index_col=0)
if len(net):
    print("-- gross, net of costs, and the attenuation grid")
    display(net)
    row = net[net["measure"] == "net of costs"]
    if len(row):
        plots.attenuation_curve(float(row["mean_bp"].iloc[0]))

### Secondary family — the WHOLE grid, winners and losers

Judged by day-blocked Romano-Wolf, resampling whole sessions jointly across variants so the
bootstrap null keeps cross-variant correlation. Reported with **best-config and median-config**
rows: a league table showing only the winner is a maximum statistic presented as a draw.

In [ ]:
league = load("g4_league", index_col=0)
bm = load("g4_best_and_median", index_col=0)
rw = load("g4_romano_wolf", index_col=0)
if len(bm):
    print("-- best-config vs median-config (the anti-selection control)")
    display(bm)
if len(league):
    print(f"-- full grid: {len(league)} variants")
    display(league.sort_values("t", ascending=False).head(15))
    plots.league_dotplot(league, highlight=set(bm["variant"]) if len(bm) else ())
if len(rw):
    print("-- Romano-Wolf family-wise adjusted p (day-blocked)")
    display(rw.head(20))
    n_sig = int((rw["p_fwer"] < 0.05).sum())
    print(f"\n  variants surviving FWER < 0.05: {n_sig} of {len(rw)}")

### Placebos

Each was specified in advance together with what its firing would mean. A placebo landing near
the reference is a **failure of the test**, not a success of the signal.

In [ ]:
plac = load("g4_placebos", index_col=0)
if len(plac):
    display(plac[[c for c in ("placebo", "expect", "mean", "t", "stars", "n",
                              "n_blocks", "hit_rate") if c in plac.columns]])
    plots.placebo_panel(plac)

## G5 — economics and capacity

Capacity is a **depth** question and this repo has no historical depth or top-of-book data, so
what follows is a traded-**volume** sensitivity and is labelled that way in its own output. It is
an upper bound on what depth would allow, not a capacity claim.

In [ ]:
cap = load("g5_capacity", index_col=0)
if len(cap):
    display(cap)

### G3b — does it also survive a basis our own curve did not produce?

Our basis control is built from the very curve that produced the signal, so on its own it
cannot separate "the ladder forecasts" from "our curve was mispriced against the futures
strip and both reverted". This second race adds a basis measured against the **Citi Velocity
swap-quote curve**. Run separately because the two bases are highly collinear — folding them
into one regression would inflate both standard errors and confound two different questions.
SR3 only: both independent sources are SOFR curves, so there is no independent ZQ basis.

In [ ]:
indep = load("g3_horse_race_independent_FUTURES", index_col=0)
if len(indep):
    display(indep)
    sig = indep[(indep["spec"] == "signal + controls") & (indep["term"] == "signal")]
    if len(sig):
        print(f"  ladder t beside the INDEPENDENT basis: {float(sig['t'].iloc[0]):+.2f}")

### The label-free cell

The identical rule driven by **unsigned print intensity** — `|delta_dv01|`, direction
ignored — so it is immune to classification accuracy. The comparison is the point. If the
signed ladder works and this does not, direction is carrying the result. If both work
similarly, the finding is a flow-**activity** effect, the direction model is carrying
nothing, and no attenuation grid rescues that.

In [ ]:
lf = load("g4_label_free", index_col=0)
if len(lf):
    display(lf)

### Conditioning splits

Terciles of block share, proximity to the 25bp policy grid, realised vol, Amihud, the
SOFR−EFFR funding spread, days to FOMC and time of day. Terciles rather than interaction
terms: at ~150 independent epochs an interaction is not identified, while *does the sign hold
in all three buckets* is answerable — and the audit asks to **segment**, not merely control.

In [ ]:
cond = load("g4_conditioning", index_col=0)
if len(cond):
    display(cond)
    consistent = cond.groupby("conditioner")["mean_net_bp"].apply(
        lambda x: bool((x > 0).all() or (x < 0).all()))
    print("\n  sign-consistent across terciles:")
    for k, v in consistent.items():
        print(f"    {'yes' if v else 'NO '}  {k}")

### G0 detail — mid offset, and flips by stratum

The offset table is the gate's real payload: our spread-to-mid minus the independent one
**is** (independent mid − our mid), so its systematic sign says which curve sits above the
other, and `share_offset_gt_quarter_bp` says how often that gap exceeds the half-spread the
direction rule is trying to read.

In [ ]:
for name in ("g0_mid_offset_bps", "g0_flip_by_curve_bucket", "g0_flip_by_hour",
             "g0_skew_vs_independent_by_hour"):
    frame = load(name, index_col=0)
    if len(frame):
        print(f"-- {name}")
        display(frame)

## Trial ledger

Every configuration evaluated, in order, including discarded ones. A family-wise correction is
only honest if the family is the one actually searched.

In [ ]:
tl = load("trial_ledger", index_col=0)
if len(tl):
    print(f"{len(tl)} trials recorded")
    display(tl)